## Successful Semantic Modelling for Power BI - attendee bootstrap (OPTIONAL)

NOT part of the normal attendee path. `0-create-lab-models` already creates the
lakehouse and SHORTCUTS the shared tables into it, which is faster and copies no
data. Do NOT run both: this notebook writes real tables called Date, Sales, ...
and the shortcut step then fails on the name clash.

Keep it as the fallback for a tenant where OneLake shortcuts are blocked. In that
case run this INSTEAD of Cell 4 of `0-create-lab-models`.

HOW TO RUN
1. Claim a numbered login (userNNNN) from the shared allocation sheet.
2. Open YOUR OWN workspace and create (or open) your own lakehouse.
3. Attach THIS notebook to your own lakehouse.
4. Set SHARED_* below to the presenter's shared lakehouse, then Run all.

WHAT IT DOES
- Copies the clean + messy Delta tables from the shared lakehouse into yours.
- Prints a checklist so you can confirm your environment is ready.

In [ ]:
# ---- CELL 1: CONFIG - point this at the presenter's shared lakehouse --------
# Ask the presenter for these two values (shown on the welcome slide).
SHARED_WORKSPACE = "Successful Semantic Modelling"     # shared workspace name
SHARED_LAKEHOUSE = "workshop_shared"                   # shared lakehouse name

# Tables every attendee needs. These match the tables the guides reference.
TABLES = ["Date", "Product", "Customer", "Territory", "Sales", "UserAccess",
          "Sales_messy", "Product_messy"]

# The Module 5 demo table is optional for attendees (presenter-driven).
INCLUDE_MODULE5_DEMO = False

In [ ]:
# ---- CELL 2: RESOLVE THE SHARED LAKEHOUSE PATH ------------------------------
# Build an ABFSS path to the shared lakehouse Tables/Files areas via OneLake.
import notebookutils

ws_id = notebookutils.lakehouse.getWithProperties(SHARED_LAKEHOUSE,
                                                  workspaceName=SHARED_WORKSPACE)["workspaceId"]
lh_id = notebookutils.lakehouse.getWithProperties(SHARED_LAKEHOUSE,
                                                  workspaceName=SHARED_WORKSPACE)["id"]
shared_root = f"abfss://{ws_id}@onelake.dfs.fabric.microsoft.com/{lh_id}"
print(f"Shared lakehouse resolved:\n  {shared_root}")

In [ ]:
# ---- CELL 3: COPY THE CLEAN TABLES INTO YOUR LAKEHOUSE ----------------------
# Read each shared Delta table and write it into your own default lakehouse.
# V-Order stays on so your copy is Direct-Lake friendly from the start.
spark.conf.set("spark.sql.parquet.vorder.enabled", "true")

tables_to_copy = TABLES + (["Sales_unhealthy"] if INCLUDE_MODULE5_DEMO else [])

for t in tables_to_copy:
    src = f"{shared_root}/Tables/{t}"
    df = spark.read.format("delta").load(src)
    (df.write.format("delta").mode("overwrite")
       .option("overwriteSchema", "true").saveAsTable(t))
    print(f"  copied {t}: {df.count():,} rows")

print("Tables copied into your lakehouse.")

# --- Faster alternative (if your tenant allows shortcuts) --------------------
# Instead of copying, create a OneLake shortcut per table. No data is duplicated.
# for t in TABLES:
#     notebookutils.fs.createShortcut(
#         name=t, path="Tables",
#         target={"oneLake": {"workspaceId": ws_id, "itemId": lh_id, "path": f"Tables/{t}"}})

In [ ]:
# ---- CELL 4: READY CHECK ----------------------------------------------------
print("\\nEnvironment check")
print("-" * 40)
ok = True
for t in TABLES:
    try:
        n = spark.table(t).count()
        print(f"  OK   {t:<12} {n:>12,} rows")
    except Exception:
        ok = False
        print(f"  MISS {t:<12} (not found - re-run cell 3)")

print("-" * 40)
print("You are ready to start Lab 1." if ok
      else "Something is missing - flag a helper before we begin.")

In [ ]:
# ---- CELL 5: HAND THE SPARK SESSION BACK ------------------------------------
# This copies data rather than shortcutting it, so it is the heaviest thing an
# attendee runs. Without this the session sits idle for 30 minutes afterwards,
# and a full room of those is capacity nobody is using.
#
# detach=False matters. detach defaults to TRUE, and on a high-concurrency
# session the default only detaches this notebook and leaves the session up.
#
# NOTE: nothing runs after this. Keep it last.
try:
    notebookutils.session.stop(detach=False)
except TypeError:
    notebookutils.session.stop()        # older runtimes have no detach parameter
except Exception as e:
    print(f"Could not stop the session ({e}). Use Stop session in the toolbar.")